# Python prompting on the situational test cases

**Research question.** Can an LLM take a natural-language description of a real situation and
construct a *formal world model* that a safe-by-design planning algorithm could later use?
The LLM here is a **neutral world modeller**, not the agent making the decision.

For the situational cases the target representation is Python: a subclass of the project's
POSG (partially observed stochastic game) abstract base class, which represents

* several players who may act simultaneously,
* state variables describing the relevant parts of the world,
* the actions available to each player,
* partial observability — different players know different things,
* stochastic transitions, and
* the full distribution `P(next_state, observation_profile | state, joint_action)`.

### The experiment

```
situation (repo markdown) + POSG ABC + prompt variant
  -> local open-weight LLM
    -> Python POSG subclass
      -> execute (subprocess, isolated)
        -> validate (the ABC's validator + a joint-action sweep)
          -> inspect quality (human review; nothing is auto-scored)
```

Every element of the world model is inferred **by the LLM**: which humans are affected and
should be players, which state variables matter for those humans' future capabilities, what
actions exist, what is uncertain and who observes what, what can happen over several steps,
and where the interaction naturally ends. No case gets a hand-written target model.

### Three levels of correctness, kept separate

| Level | Question | Decided by |
|---|---|---|
| 1. Python validity | does the code parse, import, instantiate, subclass the ABC? | `ast.parse` + a subprocess probe |
| 2. Mechanical POSG validity | do the distributions, action symbols, state shapes, observation profiles and reward vectors hold up? | the ABC's `validate()` + an added sweep over all joint actions |
| 3. Semantic world-model quality | is this a sensible model *of this situation*? | **human review** — section J prepares it, nothing is scored automatically |

A model can pass levels 1 and 2 while missing a stakeholder, inventing an actor, resolving a
genuine uncertainty by fiat, giving a player information they cannot have, or stopping after
an arbitrary step. Those failures are **surfaced, not hidden and not repaired**.

### Prompting methodology: one baseline, four single-instruction ablations

`P0_baseline` is a short six-step instruction that says what to work out before writing code.
`P1`–`P4` each add **exactly one** further instruction to P0. None of them is assumed to be an
improvement — the point is to find out. The prompt deliberately contains **no** target sizes,
no fixed horizon, and no rule for choosing a probability when one is unknown; if the model
invents unsupported numbers, that is recorded as a modelling issue rather than prevented.

### How to run

1. **Colab, with a GPU** (`Runtime -> Change runtime type -> T4 GPU`), then `Run all`. The
   default model is a 4-bit quantised open-weight coder model — no API key, no paid service.
2. **Without a GPU** every cell still runs: the notebook falls back to a stub "generation" so
   the harness can be exercised and tested. Stub rows are labelled and are not results.

Nothing outside the run directory (printed by the setup cell) is written.


## A. Setup and configuration


In [ ]:
# --- environment -------------------------------------------------------------------
import subprocess, sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/AmnaJamalKhattak/world-model-suite.git"
REPO_NAME = "world-model-suite"

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers", "accelerate", "bitsandbytes", "pandas"], check=False)
    if not Path(REPO_NAME).exists():
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)


def find_repo_root(start: Path) -> Path:
    """Walk up (then down into a fresh clone) looking for the test-case directory."""
    marker = Path("data") / "Test_Cases" / "02_situational"
    for candidate in [start, *start.parents]:
        if (candidate / marker).is_dir():
            return candidate
        if (candidate / REPO_NAME / marker).is_dir():
            return candidate / REPO_NAME
    raise FileNotFoundError(f"could not locate {marker} from {start}")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
print("repository root:", REPO_ROOT)

try:
    import torch
    HAS_GPU = torch.cuda.is_available()
    GPU_NAME = torch.cuda.get_device_name(0) if HAS_GPU else None
    GPU_GB = round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1) if HAS_GPU else 0
except Exception:
    torch, HAS_GPU, GPU_NAME, GPU_GB = None, False, None, 0
print(f"GPU: {GPU_NAME or 'none'}" + (f" ({GPU_GB} GB)" if HAS_GPU else
      "  -> Runtime > Change runtime type > T4 GPU, or run in stub mode"))


In [ ]:
# ============================ EDIT ME =============================================
# Generation backend. "local_hf" runs an open-weight model on the Colab GPU; "stub"
# exercises the harness without a model. Add a hosted backend in section E if a
# stronger model is wanted for a final run.
BACKEND = "local_hf" if HAS_GPU else "stub"

# Open-weight, ungated, Apache-2.0 code models. 7B/4-bit fits a free T4 (~5.5 GB).
MODEL_PRESETS = {
    "qwen-coder-7b":  "Qwen/Qwen2.5-Coder-7B-Instruct",    # default; free T4
    "qwen-coder-14b": "Qwen/Qwen2.5-Coder-14B-Instruct",   # needs L4/A100 to be quick
    "qwen-coder-3b":  "Qwen/Qwen2.5-Coder-3B-Instruct",    # fast smoke tests
}
MODEL_ID = MODEL_PRESETS["qwen-coder-7b"]                  # any HF instruct model id works
LOAD_IN_4BIT = True

# Experiment shape
N_GENERATIONS = 1              # independent samples per (case, prompt variant)
CASE_FILTER = ()               # e.g. ("S.1", "S.5"); empty = all 11 situational cases
VARIANT_FILTER = ()            # e.g. ("P0_baseline",); empty = all variants
ENABLE_REPAIR = True           # one mechanical-repair attempt, reported separately

# Decoding
TEMPERATURE = 0.2              # > 0 so that N_GENERATIONS > 1 gives different samples
TOP_P = 0.95
MAX_NEW_TOKENS = 2048
SEED = 0                       # per-generation seed = SEED + 1000 * generation_index

# Execution / exploration caps for the validator sweep (a cap that is hit is reported)
PROBE_TIMEOUT_S = 60
CAPS = dict(max_states=400, max_profiles=200, max_depth=12, max_samples=8, max_errors=25)
# ==================================================================================

import ast, dataclasses, datetime as dt, itertools, json, os, re, textwrap, time, types
import pandas as pd
from IPython.display import Markdown, display

POSG_REFERENCE_NB = REPO_ROOT / "notebooks" / "POSG_World_Model_Implementation.ipynb"
SITUATIONAL_DIR = REPO_ROOT / "data" / "Test_Cases" / "02_situational"
OUT_ROOT = REPO_ROOT / "experiments" / "python_posg_situational"

RUN_ID = f"{dt.datetime.now():%Y%m%d-%H%M%S}_{BACKEND}"
RUN_DIR = OUT_ROOT / RUN_ID
for sub in ("prompts", "raw", "generated", "probe", "review"):
    (RUN_DIR / sub).mkdir(parents=True, exist_ok=True)

CONFIG = dict(backend=BACKEND, model_id=MODEL_ID if BACKEND == "local_hf" else BACKEND,
              load_in_4bit=LOAD_IN_4BIT, n_generations=N_GENERATIONS,
              case_filter=list(CASE_FILTER), variant_filter=list(VARIANT_FILTER),
              enable_repair=ENABLE_REPAIR, temperature=TEMPERATURE, top_p=TOP_P,
              max_new_tokens=MAX_NEW_TOKENS, seed=SEED, caps=CAPS,
              probe_timeout_s=PROBE_TIMEOUT_S, gpu=GPU_NAME, run_id=RUN_ID)
(RUN_DIR / "config.json").write_text(json.dumps(CONFIG, indent=2, default=str))
print("backend:", BACKEND, "| model:", CONFIG["model_id"])
print("run directory:", RUN_DIR)


## B. The POSG interface

The interface below is **copied verbatim** from
`notebooks/POSG_World_Model_Implementation.ipynb` (the only POSG implementation in the
repository) so that this notebook is self-contained. It is treated as *the* interface for the
situational track, not as a settled universal interface for every category — the cell below
re-extracts it from the source notebook when that file is present and warns if the two have
diverged.

### Does this interface fit the situational task?

It covers the six things the task needs, directly:

| Requirement | Where it lives |
|---|---|
| simultaneous multi-player action | `legal_actions(s, i)` per player; `transition_kernel(s, joint_action)` |
| state variables | `state_variables` metadata + `freeze({...})` state tuples |
| per-player actions | `legal_actions(s, i)` + `action_catalogue` |
| partial observability | the observation profile returned by the kernel |
| stochastic transitions | the kernel is a distribution, not a sample |
| full `P(s', o \| s, a)` | one kernel returning `{(next_state, obs_profile): p}` |

It is stateless (state is passed in, never mutated) and small enough to implement in one
class — which is the point: fewer tokens, fewer mistakes.

**Two frictions worth knowing before reading any result**, both consequences of the
interface rather than of any model's answer:

1. `initial_state` is a **single state**, not a distribution. A situation that starts with an
   unresolved fact ("is the warning true?", "is anyone home?") has to carry that fact as a
   state variable — typically `"unknown"` — and resolve it inside a transition. The JSON track
   allows a distribution over initial states instead, so the two tracks differ structurally
   here.
2. `validate()` requires **every player to have a legal action in every non-terminal state**.
   An affected-but-passive human (a passenger, a child) therefore has to be given an explicit
   no-op to be a player at all. That interacts directly with step 1 of the prompt, which asks
   for affected humans to be included as players — expect no-op actions, and judge whether
   they are used meaningfully.

Neither is patched here. They are recorded so that a later interface revision can be argued
from evidence.


In [ ]:
# =====================================================================================
# Copied verbatim from notebooks/POSG_World_Model_Implementation.ipynb (interface cells).
# Kept as a string so the same bytes can be exec'd here, sent to the validation
# subprocess, and shown to the model — one source of truth inside this notebook.
# =====================================================================================
POSG_ABC_SOURCE = r'''from abc import ABC, abstractmethod
from typing import Any, Dict, List, Tuple, Union, Optional, FrozenSet
import functools
import math
import random

# =====================================================================
# Helper Types & Converters for Hashable States
# =====================================================================

def freeze(d: Dict[str, Any]) -> FrozenSet[Tuple[str, Any]]:
    """Converts a dictionary into a hashable frozenset of key-value pairs."""
    return frozenset(d.items())

def unfreeze(fs: FrozenSet[Tuple[str, Any]]) -> Dict[str, Any]:
    """Converts a frozenset back into a standard dictionary."""
    return dict(fs)

# =====================================================================
# 1. POSG Abstract Base Class
# =====================================================================

class AbstractPOSG(ABC):
    """
    Abstract Base Class for Partially Observable Stochastic Games (POSG).
    Supports hashable states/observations, explicit metadata, action catalogues,
    validation, schema generation, and general stochastic observation modeling.
    """

    @property
    @abstractmethod
    def players(self) -> Dict[str, Dict[str, Any]]:
        """
        Player metadata mapping player_id to descriptions/properties.
        Example: {"player_1": {"description": "First player"}}
        """
        pass

    @property
    @abstractmethod
    def state_variables(self) -> Dict[str, Dict[str, Any]]:
        """
        State-variable metadata.
        Example: {"phase": {"type": "str", "description": "Current game phase"}}
        """
        pass

    @property
    @abstractmethod
    def action_catalogue(self) -> Dict[str, Dict[str, Any]]:
        """
        Catalogue of actions and their descriptions.
        Example: {"rock": {"description": "Play rock"}}
        """
        pass

    @property
    @abstractmethod
    def initial_state(self) -> FrozenSet[Tuple[str, Any]]:
        """Returns the initial state of the game as a hashable frozenset."""
        pass

    @abstractmethod
    def legal_actions(self, state: FrozenSet[Tuple[str, Any]], player: str) -> List[str]:
        """Returns list of legal action strings for a given player and state."""
        pass

    @abstractmethod
    def transition_kernel(
        self, state: FrozenSet[Tuple[str, Any]], actions: Dict[str, str]
    ) -> Dict[Tuple[FrozenSet[Tuple[str, Any]], FrozenSet[Tuple[str, Any]]], float]:
        """
        General Stochastic Kernel: P(s', o | s, a)
        Returns mapping: (next_state, observation_profile) -> probability
        Both next_state and observation_profile are hashable frozensets.
        """
        pass

    @abstractmethod
    def rewards(
        self, state: FrozenSet[Tuple[str, Any]], actions: Dict[str, str], next_state: FrozenSet[Tuple[str, Any]]
    ) -> Dict[str, float]:
        """Returns rewards for each player."""
        pass

    @abstractmethod
    def is_terminal(self, state: FrozenSet[Tuple[str, Any]]) -> bool:
        """Returns True if state is terminal."""
        pass

    # --- Validation Mechanism ---
    def validate(self) -> List[str]:
        """
        Reliability mechanism for verifying POSG implementations.
        Catches unhashable states, invalid actions, probability sum errors, and bad metadata.
        """
        errors = []

        # 1. Declarative Metadata Checks
        if not isinstance(self.players, dict) or not self.players:
            errors.append("Validation Error: 'players' must be a non-empty dictionary.")
        if not isinstance(self.state_variables, dict):
            errors.append("Validation Error: 'state_variables' must be a dictionary.")
        if not isinstance(self.action_catalogue, dict) or not self.action_catalogue:
            errors.append("Validation Error: 'action_catalogue' must be a non-empty dictionary.")

        # 2. Initial State Verification
        try:
            init_s = self.initial_state
            hash(init_s)  # Verify hashability
        except Exception as e:
            errors.append(f"Validation Error: Initial state is invalid or unhashable: {e}")
            return errors

        # 3. Reachability & Kernel Distribution Checks
        visited_states = set()
        queue = [init_s]

        while queue:
            curr_state = queue.pop(0)
            if curr_state in visited_states:
                continue
            visited_states.add(curr_state)

            if self.is_terminal(curr_state):
                continue

            # Validate legal actions
            player_actions = {}
            for p in self.players.keys():
                leg_acts = self.legal_actions(curr_state, p)
                if not leg_acts:
                    errors.append(f"Validation Error: Player '{p}' has no legal actions in state {dict(curr_state)}.")
                for act in leg_acts:
                    if act not in self.action_catalogue:
                        errors.append(f"Validation Error: Undeclared action '{act}' for player '{p}'.")
                player_actions[p] = leg_acts

            # Sample action profile to validate kernel distribution
            if len(player_actions) == len(self.players):
                sample_joint_action = {p: acts[0] for p, acts in player_actions.items() if acts}
                if len(sample_joint_action) == len(self.players):
                    try:
                        dist = self.transition_kernel(curr_state, sample_joint_action)
                        prob_sum = sum(dist.values())
                        
                        if not math.isclose(prob_sum, 1.0, abs_tol=1e-5):
                            errors.append(f"Validation Error: Probabilities sum to {prob_sum} (expected 1.0) for state {dict(curr_state)}.")

                        for (next_s, obs_prof), prob in dist.items():
                            hash(next_s)
                            hash(obs_prof)
                            
                            if prob < 0.0 or prob > 1.0:
                                errors.append(f"Validation Error: Invalid probability {prob} in transition distribution.")
                            
                            obs_dict = dict(obs_prof)
                            if set(obs_dict.keys()) != set(self.players.keys()):
                                errors.append("Validation Error: Observation profile players mismatch declared players.")
                            
                            if next_s not in visited_states:
                                queue.append(next_s)
                    except Exception as e:
                        errors.append(f"Validation Error during kernel execution at state {dict(curr_state)}: {e}")

        return errors

    # --- Schema Generation ---
    def schema(self) -> str:
        """Generates a human/LLM-readable structural schema of the game."""
        lines = ["=== POSG Game Schema ===", "\nPlayers:"]
        for p, meta in self.players.items():
            lines.append(f"  - {p}: {meta.get('description', 'No description')}")

        lines.append("\nState Variables:")
        for sv, meta in self.state_variables.items():
            lines.append(f"  - {sv} ({meta.get('type', 'Any')}): {meta.get('description', '')}")

        lines.append("\nAction Catalogue:")
        for act, meta in self.action_catalogue.items():
            lines.append(f"  - {act}: {meta.get('description', '')}")

        lines.append(f"\nInitial State: {dict(self.initial_state)}")
        return "\n".join(lines)'''


def load_module(src: str, name: str) -> types.ModuleType:
    mod = types.ModuleType(name)
    mod.__file__ = f"<{name}>"
    exec(compile(src, f"<{name}>", "exec"), mod.__dict__)
    return mod


posg_abc = load_module(POSG_ABC_SOURCE, "posg_abc_inline")
AbstractPOSG, freeze, unfreeze = posg_abc.AbstractPOSG, posg_abc.freeze, posg_abc.unfreeze

ABC_PATH = RUN_DIR / "posg_abc.py"
ABC_PATH.write_text("# copied from notebooks/POSG_World_Model_Implementation.ipynb\n\n"
                    + POSG_ABC_SOURCE)

print("abstract methods:", sorted(AbstractPOSG.__abstractmethods__))
print("provided        :", ["freeze", "unfreeze", "validate", "schema"])


def _extract_abc_from_notebook(nb_path: Path) -> str:
    nb = json.loads(nb_path.read_text())
    wanted = [s for s in ("".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code")
              if "from abc import" in s or "def freeze(" in s or "class AbstractPOSG" in s]
    src = "\n\n".join(wanted)
    return "\n".join(l for l in src.splitlines()
                     if "pettingzoo" not in l and "gymnasium" not in l)


def _normalise(src: str) -> str:
    """Compare interface text ignoring trailing whitespace: Jupyter strips it from cell
    source, so a byte comparison would flag a difference this notebook cannot avoid."""
    return "\n".join(line.rstrip() for line in src.strip().splitlines())


if POSG_REFERENCE_NB.exists():                      # drift check against the source notebook
    _upstream = _extract_abc_from_notebook(POSG_REFERENCE_NB)
    print("copy matches the source notebook:",
          "yes" if _normalise(_upstream) == _normalise(POSG_ABC_SOURCE) else
          "NO — the source notebook changed; re-copy it before trusting any comparison")


In [ ]:
# The two reference implementations from the same notebook, used only as harness fixtures
# in section F (never shown to the model).
REFERENCE_IMPLS = {}
if POSG_REFERENCE_NB.exists():
    _cells = ["".join(c["source"]) for c in json.loads(POSG_REFERENCE_NB.read_text())["cells"]
              if c["cell_type"] == "code"]
    for _key, _needle in (("reference_rps", "class RockPaperScissorsPOSG"),
                          ("reference_dice", "class DiceDuelPOSG")):
        _hit = next((s for s in _cells if _needle in s), None)
        if _hit:
            REFERENCE_IMPLS[_key] = _hit
print("reference implementations available:", list(REFERENCE_IMPLS))


## C. The situational test cases

The eleven situational cases are read from `data/Test_Cases/02_situational/*.md`. Each file
has an `### LLM Input` section — the situation, passed to the model **verbatim**, so results
stay comparable with the JSON track — and a `### Judging Criteria` section listing what a good
model *should* and *should not* contain.

The judging criteria are the repository's own annotations. They are **never shown to the
model**; they appear only in the section-J review sheets, as the reference a human reviewer
reads against. No target world model is written here for any case.

`data/README.md` classifies each case by expected output format. For the situational track,
**S2, S5 and S7 are "Both"** (a JSON skeleton is feasible) and **S1, S3, S4, S6, S8, S9, S10,
S11 are "Python"** (open-ended state space, JSON enumeration impractical). All eleven are run
here; the classification is carried into the results table so the two groups can be compared.


In [ ]:
_HEADING = re.compile(r"^##\s*Test Case\s+(S\.\d+)\s*:\s*(.+?)\s*$", re.M)
_SECTION = re.compile(r"^###\s+(.*?)\s*$", re.M)
_NOT_LINE = re.compile(r"should\s*\**\s*NOT\s+include", re.I)
_BULLET = re.compile(r"^\s*[-*]\s+(.*\S)\s*$")

# from data/README.md, "Output-Format Classification Table"
FORMAT_CLASS = {"S.2": "Both", "S.5": "Both", "S.7": "Both"}


def _clean(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip().rstrip(";").strip()


def parse_case(path: Path) -> dict:
    md = path.read_text()
    head = _HEADING.search(md)
    if not head:
        raise ValueError(f"{path.name}: no '## Test Case S.N: ...' heading")

    marks = list(_SECTION.finditer(md))
    sections = {}
    for i, m in enumerate(marks):
        end = marks[i + 1].start() if i + 1 < len(marks) else len(md)
        sections[m.group(1).strip().lower()] = md[m.end():end].strip()

    lines = sections.get("judging criteria", "").splitlines()
    split_at = next((i for i, l in enumerate(lines) if _NOT_LINE.search(l)), len(lines))
    bullets = lambda block: [_clean(m.group(1)) for line in block
                             for m in [_BULLET.match(line)] if m]

    case = {"case_id": head.group(1), "title": head.group(2),
            "situation": _clean(sections.get("llm input", "")),
            "expect": bullets(lines[:split_at]), "avoid": bullets(lines[split_at:]),
            "format_class": FORMAT_CLASS.get(head.group(1), "Python"),
            "source_file": str(path.relative_to(REPO_ROOT))}
    if not case["situation"]:
        raise ValueError(f"{path.name}: empty '### LLM Input' section")
    return case


CASES = sorted((parse_case(p) for p in SITUATIONAL_DIR.glob("*.md")),
               key=lambda c: int(c["case_id"].split(".")[1]))
if CASE_FILTER:
    CASES = [c for c in CASES if c["case_id"] in CASE_FILTER]

pd.DataFrame([{"case": c["case_id"], "title": c["title"], "format_class": c["format_class"],
               "situation_words": len(c["situation"].split()),
               "expect_bullets": len(c["expect"]), "avoid_bullets": len(c["avoid"])}
              for c in CASES])


## D. Prompts: one baseline, four single-instruction ablations

The system message is **constant across variants**: a one-line role, the interface, and the
mechanical contract the validator enforces. Those are facts about the interface, not modelling
advice — without them nothing would validate and the experiment would measure the wrong thing.

The user message is the situation plus the variant's instruction list. `P0_baseline` is the
six-step baseline; each of `P1`–`P4` is P0 **plus exactly one sentence**:

| Variant | Added to P0 | What it tests |
|---|---|---|
| `P0_baseline` | — | can the model do this at all, unprompted about how? |
| `P1_assumptions` | avoid unfounded assumptions | does discouraging invention reduce invented actors/details — or reduce useful commonsense inference? |
| `P2_uncertainty` | leave important unspecified facts uncertain | does it stop uncertainties being resolved by fiat — or just add unused branches? |
| `P3_observability` | distinguish the actual state from what each player observes | does it improve observation profiles — or add noise? |
| `P4_horizon` | choose a sufficient but tractable horizon | does it change how the model ends — or produce arbitrary cut-offs anyway? |

**None of these is assumed to be an improvement.** An addition that lowers validity or makes
models worse is a result.

What the prompt deliberately does **not** contain: target numbers of players, state variables
or actions; a fixed number of steps; any rule for picking a probability when the situation does
not give one. If a model writes `0.3` for something the scenario never quantified, section G
records it and section J shows it to the reviewer.


In [ ]:
def interface_excerpt(src: str) -> str:
    """Signatures and docstrings only — what an implementer needs, without the machinery."""
    provided = {"validate": "Provided by the base class. Returns a list of error strings.",
                "schema": "Provided by the base class. Renders the declared metadata."}
    tree, out = ast.parse(src), []
    for node in tree.body:
        if isinstance(node, ast.FunctionDef) and node.name in ("freeze", "unfreeze"):
            out.append(ast.unparse(node))
        elif isinstance(node, ast.ClassDef) and node.name == "AbstractPOSG":
            body = []
            for item in node.body:
                if isinstance(item, ast.Expr) and isinstance(item.value, ast.Constant):
                    body.append(item)
                elif isinstance(item, ast.FunctionDef):
                    decorators = " ".join(ast.unparse(d) for d in item.decorator_list)
                    if "abstractmethod" in decorators:
                        item.body = [ast.Expr(ast.Constant(ast.get_docstring(item) or "..."))]
                        body.append(item)
                    elif item.name in provided:
                        item.decorator_list = []
                        item.body = [ast.Expr(ast.Constant(provided[item.name]))]
                        body.append(item)
            node.body = body
            out.append(ast.unparse(node))
    return "\n\n".join(out)


INTERFACE_EXCERPT = interface_excerpt(POSG_ABC_SOURCE)

# Facts about the interface that its signatures do not show, taken from validate().
MECHANICAL_CONTRACT = """\
Interface rules the validator checks:
- Players act simultaneously. legal_actions(state, player) must return at least one action for
  every player in every non-terminal state, and [] exactly when is_terminal(state) is True.
- Every action returned must be a key of action_catalogue.
- Every state is freeze({...}) over the declared state_variables, with the same keys in every
  state and hashable values.
- transition_kernel returns {(next_state, observation_profile): probability} with the
  probabilities summing to 1. observation_profile is freeze({player: observation}) with exactly
  one entry per declared player.
- rewards returns one float per declared player. This project derives value downstream, so the
  values are not evaluated.
- initial_state is a single state, not a distribution.
- AbstractPOSG, freeze and unfreeze are already imported. Do not redefine or re-import them."""

SYSTEM_PROMPT = f"""You build formal world models. You are not the agent in the situation and you \
do not decide what should be done: you describe who can act, what can happen, and who knows what.

The interface to implement:

```python
{INTERFACE_EXCERPT}
```

{MECHANICAL_CONTRACT}"""

# --- P0: the baseline instruction list -------------------------------------------------
P0_STEPS = """\
1. Identify the humans likely to be affected by what happens in this situation, including
   relevant humans who are not explicitly named, and include them as players alongside the
   non-human agent.
2. Identify the aspects of the world that matter for those humans' ability to achieve a wide
   range of possible future goals, and represent them as state variables.
3. Identify the possible behaviours and actions of the players.
4. Consider the possible consequences and developments over several steps.
5. Decide on natural terminal states.
6. Only then implement the POSG subclass.

Return Python code only: one class SituationPOSG(AbstractPOSG), plus any module-level constants
it needs. No explanation, no markdown fences, no example usage."""

# --- one added instruction per ablation ------------------------------------------------
ADDENDA = {
    "P1_assumptions":
        "Avoid unfounded assumptions: do not build in details the situation does not support.",
    "P2_uncertainty":
        "Important facts the situation does not specify should remain uncertain in the model "
        "rather than being resolved arbitrarily.",
    "P3_observability":
        "Explicitly distinguish the actual state of the world from what each player observes "
        "or knows.",
    "P4_horizon":
        "Choose a horizon and terminal structure that is sufficient for the situation and still "
        "tractable, rather than an arbitrary number of steps.",
}

PROMPT_VARIANTS = {"P0_baseline": None, **{k: k for k in ADDENDA}}
if VARIANT_FILTER:
    PROMPT_VARIANTS = {k: v for k, v in PROMPT_VARIANTS.items() if k in VARIANT_FILTER}


def build_prompt(case: dict, variant: str) -> tuple:
    """(system, user). Only the user message differs between variants."""
    steps = P0_STEPS
    if PROMPT_VARIANTS.get(variant):
        addendum = ADDENDA[PROMPT_VARIANTS[variant]]
        steps = P0_STEPS.replace("6. Only then implement the POSG subclass.",
                                 f"6. {addendum}\n7. Only then implement the POSG subclass.")
    return SYSTEM_PROMPT, f"SITUATION:\n\n{case['situation']}\n\n{steps}"


REPAIR_INSTRUCTION = """\
The class you produced does not satisfy the interface. Reported problems:

{errors}

Return the complete corrected class. Change only what these problems require: keep the same
players, actions, state variables and uncertainties. Python code only."""

print("variants:", list(PROMPT_VARIANTS))
_s, _u = build_prompt(CASES[0], "P3_observability")
print(f"\nsystem: {len(_s)} chars (~{len(_s)//4} tokens) | user: {len(_u)} chars "
      f"(~{len(_u)//4} tokens)\n")
print("--- user message (P3_observability) ---\n")
print(_u)


## E. Generation backends

`generate(system, user, seed) -> (text, meta)` is the only model-specific code. Two backends
ship here; a hosted one can be added by writing a third class with the same method — nothing
else in the notebook changes.

* **`LocalHFGenerator`** — an open-weight instruct model from Hugging Face, 4-bit quantised
  (NF4, fp16 compute) so a 7B coder model fits a free Colab T4 in about 5.5 GB. Default:
  `Qwen/Qwen2.5-Coder-7B-Instruct` (Apache-2.0, ungated, strong at Python). `MODEL_PRESETS`
  also lists a 3B for smoke tests and a 14B for an L4/A100 session.
* **`StubGenerator`** — returns a fixed, deliberately thin implementation. It exists to test
  the harness (including one broken variant, so the repair path runs) when there is no GPU.
  Its rows are labelled `stub` and are not experimental results.

Sampling is on (`TEMPERATURE = 0.2`) so that `N_GENERATIONS > 1` produces genuinely different
samples; the seed is `SEED + 1000 * generation_index`, recorded per attempt, so any single
generation can be reproduced.


In [ ]:
class StubGenerator:
    """Fixture for exercising the harness without a model. Not an experimental result."""

    name = "stub"
    BODY = '''\
P_HAZARD = 0.5

class SituationPOSG(AbstractPOSG):
    """STUB — a fixture for testing the harness, not a generated world model."""

    @property
    def players(self):
        return {"robot": {"description": "The non-human agent."},
                "human": {"description": "The one human this stub represents."}}

    @property
    def state_variables(self):
        return {"phase": {"type": "str", "description": "start | acted | done"},
                "hazard": {"type": "str", "description": "unknown | real | absent"}}

    @property
    def action_catalogue(self):
        return {"check": {"description": "Gather information."},
                "act": {"description": "Take the task action."},
                "wait": {"description": "Do nothing this step."}}

    @property
    def initial_state(self):
        return freeze({"phase": "start", "hazard": "unknown"})

    def legal_actions(self, state, player):
        if self.is_terminal(state):
            return []
        return ["check", "act", "wait"] if player == "robot" else ["wait"]

    def transition_kernel(self, state, actions):
        s = unfreeze(state)
        if actions["robot"] == "check" and s["hazard"] == "unknown":
            out = {}
            for value, p in (("real", P_HAZARD), ("absent", 1.0 - P_HAZARD)):
                nxt = freeze({"phase": "acted", "hazard": value})
                obs = freeze({"robot": "observed_" + value, "human": "robot_checked"})
                out[(nxt, obs)] = p
            return out
        phase = "done" if actions["robot"] == "act" else "acted"
        nxt = freeze({"phase": phase, "hazard": s["hazard"]})
        obs = freeze({"robot": "did_" + actions["robot"], "human": "nothing_new"})
        return {(nxt, obs): 1.0}

    def rewards(self, state, actions, next_state):
        return {"robot": 0.0, "human": 0.0}

    def is_terminal(self, state):
        return unfreeze(state)["phase"] == "done"
'''

    def generate(self, system: str, user: str, seed: int) -> tuple:
        # fail on roughly a quarter of the prompts, deterministically, so the tables show a
        # mix and the repair path runs; a repair prompt always yields the good version
        is_repair = "Reported problems:" in user
        broken = not is_repair and sum(map(ord, user)) % 4 == 0
        code = self.BODY.replace("1.0 - P_HAZARD", "0.3") if broken else self.BODY
        return f"```python\n{code}```", {"backend": "stub", "model_id": "stub",
                                         "input_tokens": None, "output_tokens": None,
                                         "seconds": 0.0, "seed": seed}


class LocalHFGenerator:
    """Open-weight instruct model on the local GPU, 4-bit quantised."""

    name = "local_hf"

    def __init__(self, model_id: str, load_in_4bit: bool = True):
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
        self.model_id = model_id
        print(f"loading {model_id} (4-bit={load_in_4bit}) — first run downloads several GB ...")
        t0 = time.time()
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                   bnb_4bit_compute_dtype=torch.float16,
                                   bnb_4bit_use_double_quant=True) if load_in_4bit else None
        kwargs = dict(device_map="auto", quantization_config=quant)
        try:                                    # `dtype` in newer transformers, `torch_dtype` before
            self.model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, **kwargs)
        except TypeError:
            self.model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16,
                                                              **kwargs)
        self.model.eval()
        print(f"loaded in {time.time() - t0:.0f}s")

    def generate(self, system: str, user: str, seed: int) -> tuple:
        messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
        try:
            text = self.tokenizer.apply_chat_template(messages, tokenize=False,
                                                      add_generation_prompt=True)
        except Exception:      # some chat templates have no system role — fold it into the user turn
            text = self.tokenizer.apply_chat_template(
                [{"role": "user", "content": f"{system}\n\n---\n\n{user}"}],
                tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)
        torch.manual_seed(seed)
        t0 = time.time()
        with torch.no_grad():
            out = self.model.generate(
                **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=TEMPERATURE > 0,
                temperature=TEMPERATURE if TEMPERATURE > 0 else None,
                top_p=TOP_P if TEMPERATURE > 0 else None, repetition_penalty=1.05,
                pad_token_id=self.tokenizer.eos_token_id)
        n_in = inputs["input_ids"].shape[-1]
        n_out = int(out.shape[-1] - n_in)
        completion = self.tokenizer.decode(out[0][n_in:], skip_special_tokens=True)
        return completion, {"backend": "local_hf", "model_id": self.model_id,
                            "input_tokens": int(n_in), "output_tokens": n_out,
                            "truncated": n_out >= MAX_NEW_TOKENS,   # ran out of budget mid-answer
                            "seconds": round(time.time() - t0, 1), "seed": seed}


# A hosted backend would go here: same generate(system, user, seed) -> (text, meta) signature.
# Deliberately not implemented — methodology development runs on the open-weight model.

GENERATOR = LocalHFGenerator(MODEL_ID, LOAD_IN_4BIT) if BACKEND == "local_hf" else StubGenerator()
if BACKEND != "local_hf":
    print("!! stub backend: the pipeline runs, but the rows below are NOT experimental results.")


In [ ]:
CODE_FENCE = re.compile(r"```(?:python|py)?\s*\n(.*?)(?:```|\Z)", re.S)


def extract_python(text: str) -> str:
    """Pull the implementation out of the response, fenced or not, complete or truncated."""
    blocks = CODE_FENCE.findall(text or "")
    if blocks:
        return max(blocks, key=len).strip()
    if re.search(r"^\s*(class\s+\w+\s*\(|[A-Z_]+\s*=|import |from )", text or "", re.M):
        return (text or "").strip()
    return ""


## F. Execution and mechanical validation

Generated code runs in a **subprocess** with a timeout. That matters more here than usual: the
quantised model occupies the GPU in this process, and a runaway state space or an infinite loop
in generated code would otherwise cost the whole session. The probe reports, in order:

1. **import** — the module loads (syntax is checked first, in-process, with `ast.parse`);
2. **class** — it defines a subclass of `AbstractPOSG`;
3. **instantiate** — no abstract method left unimplemented;
4. **validate** — the ABC's own `validate()`, unchanged;
5. **sweep** — added here.

The sweep exists because `validate()` follows only the *first* legal action profile out of each
state, and never looks at rewards or at whether states keep a consistent shape. The sweep walks
the reachable graph over the **full joint-action product** and additionally checks that every
state carries exactly the declared state variables, that reward vectors have one number per
player, and that observation profiles cover every player — while collecting what section J
needs for review: the actions each player actually has, the observations each player actually
sees, the terminal states reached, and every probability value used.

Caps bound the walk. **A cap that is hit is reported, never hidden**: it means the model is too
large to check exhaustively, which is itself a finding about the generated model.


In [ ]:
PROBE_SOURCE = r'''
"""Execute one generated POSG module in a fresh interpreter and report on it.

Usage:  python probe.py <posg_abc.py> <module.py> <caps.json>
Always prints a single JSON object on stdout, whatever goes wrong.
"""
from __future__ import annotations

import importlib.util
import io
import itertools
import json
import math
import sys
import traceback


def _jsonify(value):
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    if isinstance(value, frozenset):
        try:
            return {str(k): _jsonify(v) for k, v in dict(value).items()}
        except Exception:
            return sorted(repr(v) for v in value)
    if isinstance(value, dict):
        return {str(k): _jsonify(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [_jsonify(v) for v in value]
    return repr(value)


def _err(exc):
    return {"type": type(exc).__name__, "message": str(exc)[:2000],
            "traceback": "".join(traceback.format_exception(
                type(exc), exc, exc.__traceback__))[-4000:]}


def _load(path, name):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


def sweep(game, caps):
    """Walk the reachable state graph over all joint actions, within caps."""
    errors, warnings, caps_hit = [], [], []
    players = list(game.players.keys())
    declared_vars = set(game.state_variables.keys())
    catalogue = set(game.action_catalogue.keys())

    probabilities = set()
    actions_by_player = {p: set() for p in players}
    observations = {p: set() for p in players}
    terminals, reward_samples, n_entries = [], [], 0
    seen, frontier, max_depth = set(), [(game.initial_state, 0)], 0

    def note(msg):
        if msg not in errors and len(errors) < caps["max_errors"]:
            errors.append(msg)

    while frontier:
        state, depth = frontier.pop(0)
        try:
            if state in seen:
                continue
            seen.add(state)
        except TypeError as exc:
            note("state is not hashable: %s (%s)" % (_jsonify(state), exc))
            continue
        if len(seen) > caps["max_states"]:
            caps_hit.append("max_states")
            break
        max_depth = max(max_depth, depth)
        label = _jsonify(state)

        if not isinstance(state, frozenset):
            note("state is %s, not a frozenset: %s" % (type(state).__name__, label))
            continue
        try:
            keys = set(dict(state).keys())
        except Exception as exc:
            note("state is not a frozenset of (name, value) pairs: %s (%s)" % (label, exc))
            continue
        if keys != declared_vars:
            note("state variables disagree with the declared catalogue: missing=%s undeclared=%s in %s"
                 % (sorted(declared_vars - keys), sorted(keys - declared_vars), label))

        try:
            terminal = game.is_terminal(state)
        except Exception as exc:
            note("is_terminal raised %s in %s: %s" % (type(exc).__name__, label, exc))
            continue
        if terminal:
            if len(terminals) < caps["max_samples"]:
                terminals.append(label)
            continue

        legal = {}
        for p in players:
            try:
                acts = game.legal_actions(state, p)
            except Exception as exc:
                note("legal_actions raised %s for player '%s' in %s: %s"
                     % (type(exc).__name__, p, label, exc))
                acts = []
            if not acts:
                note("player '%s' has no legal actions in non-terminal state %s" % (p, label))
            for a in acts:
                if a not in catalogue:
                    note("action '%s' for player '%s' is not in the action catalogue" % (a, p))
            actions_by_player.setdefault(p, set()).update(acts)
            legal[p] = list(acts)
        if any(not v for v in legal.values()):
            continue

        profiles = list(itertools.product(*(legal[p] for p in players)))
        if len(profiles) > caps["max_profiles"]:
            caps_hit.append("max_profiles")
            profiles = profiles[:caps["max_profiles"]]

        for profile in profiles:
            actions = dict(zip(players, profile))
            try:
                dist = game.transition_kernel(state, actions)
            except Exception as exc:
                note("transition_kernel raised %s in %s under %s: %s"
                     % (type(exc).__name__, label, actions, exc))
                continue
            if not isinstance(dist, dict) or not dist:
                note("transition_kernel returned an empty/invalid distribution in %s under %s"
                     % (label, actions))
                continue

            total = 0.0
            for key, prob in dist.items():
                n_entries += 1
                if not (isinstance(key, tuple) and len(key) == 2):
                    note("kernel key is not a (next_state, observation_profile) pair: %s" % _jsonify(key))
                    continue
                next_state, obs_profile = key
                if not isinstance(prob, (int, float)) or isinstance(prob, bool):
                    note("probability %r is not a number in %s under %s" % (prob, label, actions))
                    continue
                if prob < 0 or prob > 1:
                    note("probability %s outside [0, 1] in %s under %s" % (prob, label, actions))
                total += float(prob)
                probabilities.add(round(float(prob), 6))

                try:
                    obs = dict(obs_profile)
                except Exception:
                    note("observation profile is not a frozenset of (player, observation) pairs: %s"
                         % _jsonify(obs_profile))
                    obs = {}
                if set(obs.keys()) != set(players):
                    note("observation profile players %s != declared players %s"
                         % (sorted(map(str, obs.keys())), sorted(players)))
                for p, o in obs.items():
                    bucket = observations.setdefault(str(p), set())
                    if len(bucket) < caps["max_samples"]:
                        bucket.add(repr(o))

                try:
                    rew = game.rewards(state, actions, next_state)
                except Exception as exc:
                    note("rewards raised %s in %s under %s: %s"
                         % (type(exc).__name__, label, actions, exc))
                    rew = None
                if rew is not None:
                    if not isinstance(rew, dict) or set(rew.keys()) != set(players):
                        note("rewards keys %s != declared players %s"
                             % (sorted(map(str, rew)) if isinstance(rew, dict) else repr(rew),
                                sorted(players)))
                    else:
                        for p, r in rew.items():
                            if not isinstance(r, (int, float)) or isinstance(r, bool):
                                note("reward %r for player '%s' is not a number" % (r, p))
                        if len(reward_samples) < caps["max_samples"]:
                            reward_samples.append({"actions": _jsonify(actions),
                                                   "rewards": _jsonify(rew)})

                if next_state not in seen:
                    if len(seen) + len(frontier) < caps["max_states"]:
                        frontier.append((next_state, depth + 1))
                    else:
                        caps_hit.append("max_states")

            if not math.isclose(total, 1.0, abs_tol=1e-5):
                note("probabilities sum to %.6f (expected 1.0) in %s under %s"
                     % (total, label, actions))

        if depth >= caps["max_depth"]:
            caps_hit.append("max_depth")
            frontier = [(s, d) for s, d in frontier if d <= caps["max_depth"]]

    if not terminals:
        warnings.append("no terminal state was reached within the exploration caps")
    used = set()
    for acts in actions_by_player.values():
        used |= acts
    unused = sorted(catalogue - used)
    if unused:
        warnings.append("declared in the catalogue but never legal: %s" % unused)

    summary = {"errors": errors, "warnings": warnings, "caps_hit": sorted(set(caps_hit)),
               "n_states_explored": len(seen), "n_terminal_sampled": len(terminals),
               "max_depth": max_depth, "n_transition_entries": n_entries,
               "probabilities": sorted(probabilities)}
    structure = {"terminal_states": terminals,
                 "observations": {p: sorted(v) for p, v in observations.items()},
                 "actions_by_player": {p: sorted(v) for p, v in actions_by_player.items()},
                 "reward_samples": reward_samples, "unused_catalogue_actions": unused}
    return summary, structure


def main():
    out = {"stage": "start", "ok": False, "error": None, "class_name": None,
           "candidate_classes": [], "abc_compliant": False, "validate_errors": None,
           "sweep": None, "structure": None, "captured_stdout": ""}
    # generated code may print on import; keep stdout clean so the JSON stays parseable
    real_stdout, sys.stdout = sys.stdout, io.StringIO()
    try:
        abc_path, module_path, caps_json = sys.argv[1], sys.argv[2], sys.argv[3]
        caps = json.loads(caps_json)

        out["stage"] = "import_abc"
        AbstractPOSG = _load(abc_path, "posg_abc").AbstractPOSG

        out["stage"] = "import_module"
        mod = _load(module_path, "generated_posg")

        out["stage"] = "find_class"
        candidates = [obj for obj in vars(mod).values()
                      if isinstance(obj, type) and issubclass(obj, AbstractPOSG)
                      and obj is not AbstractPOSG and obj.__module__ == "generated_posg"]
        out["candidate_classes"] = [c.__name__ for c in candidates]
        if not candidates:
            raise TypeError("the module defines no subclass of AbstractPOSG")
        cls = candidates[0]
        out["class_name"] = cls.__name__

        out["stage"] = "instantiate"
        game = cls()
        out["abc_compliant"] = True

        out["stage"] = "validate"
        try:
            out["validate_errors"] = list(game.validate())
        except Exception as exc:
            # validate() propagates exceptions from is_terminal/legal_actions; keep going so
            # the sweep can still say *where* the implementation breaks.
            out["validate_errors"] = ["validate() raised %s: %s" % (type(exc).__name__, exc)]

        out["stage"] = "sweep"
        summary, structure = sweep(game, caps)
        out["sweep"] = summary

        out["stage"] = "structure"

        def _initial_actions(p):
            try:
                return list(game.legal_actions(game.initial_state, p))
            except Exception as exc:
                return ["<error: %s: %s>" % (type(exc).__name__, exc)]

        structure.update({
            "players": {str(p): (m.get("description") if isinstance(m, dict) else _jsonify(m))
                        for p, m in game.players.items()},
            "state_variables": {str(v): _jsonify(m) for v, m in game.state_variables.items()},
            "action_catalogue": {str(a): (m.get("description") if isinstance(m, dict) else _jsonify(m))
                                 for a, m in game.action_catalogue.items()},
            "initial_state": _jsonify(game.initial_state),
            "legal_actions_initial": {str(p): _initial_actions(p) for p in game.players},
        })
        out["structure"] = structure
        out["stage"] = "done"
        out["ok"] = True
    except BaseException as exc:
        out["error"] = _err(exc)
    finally:
        out["captured_stdout"] = sys.stdout.getvalue()[-2000:]
        sys.stdout = real_stdout
    print(json.dumps(out, default=str))


if __name__ == "__main__":
    main()
'''

PROBE_PATH = RUN_DIR / "probe.py"
PROBE_PATH.write_text(PROBE_SOURCE)

MODULE_PREAMBLE = ("# generated by python_prompting_situational_tests.ipynb\n"
                   "from posg_abc import AbstractPOSG, freeze, unfreeze\n"
                   "from typing import Any, Dict, FrozenSet, List, Optional, Tuple\n"
                   "import itertools, math, random\n\n")


def write_module(code: str, path: Path) -> Path:
    path.write_text(MODULE_PREAMBLE + code.rstrip() + "\n")
    return path


def run_probe(module_path: Path) -> dict:
    """Import, instantiate, validate and sweep the module in a fresh interpreter."""
    try:
        proc = subprocess.run(
            [sys.executable, str(PROBE_PATH), str(ABC_PATH), str(module_path), json.dumps(CAPS)],
            capture_output=True, text=True, timeout=PROBE_TIMEOUT_S, cwd=str(RUN_DIR),
            env={**os.environ, "PYTHONDONTWRITEBYTECODE": "1"})
    except subprocess.TimeoutExpired:
        return {"stage": "timeout", "ok": False, "error": {
            "type": "Timeout", "message": f"probe exceeded {PROBE_TIMEOUT_S}s "
            "(runaway state space or non-terminating code)", "traceback": ""}}
    try:
        return json.loads(proc.stdout)
    except json.JSONDecodeError:
        return {"stage": "probe_crash", "ok": False, "error": {
            "type": "ProbeCrash", "message": (proc.stderr or proc.stdout)[-2000:], "traceback": ""}}


def mechanical_verdict(code: str, probe: dict) -> dict:
    """Flatten probe output into the levels reported in the results tables."""
    syntax_ok, syntax_error = True, ""
    if code:
        try:
            ast.parse(code)
        except SyntaxError as exc:
            syntax_ok, syntax_error = False, f"SyntaxError: {exc}"
    else:
        syntax_ok, syntax_error = False, "no code in the response"

    stage = probe.get("stage")
    err = probe.get("error") or {}
    validate_errors = probe.get("validate_errors") or []
    sweep = probe.get("sweep") or {}
    sweep_errors = sweep.get("errors") or []

    import_ok = syntax_ok and stage not in ("import_module", "import_abc", "probe_crash",
                                            "timeout", "no_code")
    class_ok = bool(probe.get("class_name"))
    abc_ok = bool(probe.get("abc_compliant"))          # imports, subclasses, instantiates
    validate_ok = abc_ok and probe.get("validate_errors") is not None and not validate_errors
    sweep_ok = probe.get("sweep") is not None and not sweep_errors
    verdict = {"syntax_ok": syntax_ok, "import_ok": import_ok, "class_found": class_ok,
               "exec_ok": abc_ok, "validate_ok": validate_ok, "sweep_ok": sweep_ok,
               "mechanical_ok": bool(syntax_ok and import_ok and class_ok and abc_ok
                                     and validate_ok and sweep_ok),
               "failed_stage": None, "errors": []}
    if not syntax_ok:
        verdict["failed_stage"], verdict["errors"] = "syntax", [syntax_error]
    elif not import_ok or not class_ok or not abc_ok:
        verdict["failed_stage"] = stage
        verdict["errors"] = [f"{err.get('type', 'Error')}: {err.get('message', '')}".strip()]
    elif validate_errors:
        verdict["failed_stage"], verdict["errors"] = "validate", validate_errors
    elif sweep_errors:
        verdict["failed_stage"], verdict["errors"] = "sweep", sweep_errors
    elif err:
        verdict["failed_stage"] = stage
        verdict["errors"] = [f"{err.get('type', 'Error')}: {err.get('message', '')}".strip()]
        verdict["mechanical_ok"] = False
    return verdict


### F.1 Harness self-test (no model required)

The two reference implementations from the interface notebook go through the harness first,
alongside a copy of Rock-Paper-Scissors mutated so its probabilities sum to 0.9. The references
must pass and the mutation must be caught. If this prints FAILED, nothing below it is
trustworthy. Same negative-testing idea as the 17-mutation self-test in the JSON notebook.


In [ ]:
if REFERENCE_IMPLS:
    _fixtures = dict(REFERENCE_IMPLS)
    _fixtures["mutated_rps_sum_0.9"] = _fixtures["reference_rps"].replace(
        "return {(next_state, obs_profile): 1.0}", "return {(next_state, obs_profile): 0.9}")
    assert _fixtures["mutated_rps_sum_0.9"] != _fixtures["reference_rps"], \
        "the mutation matched nothing — the reference implementation changed; fix the self-test"

    _rows, _expected = [], {"reference_rps": True, "reference_dice": True,
                            "mutated_rps_sum_0.9": False}
    for _name, _src in _fixtures.items():
        _p = run_probe(write_module(_src, RUN_DIR / "generated" / f"_selftest_{_name}.py"))
        _v = mechanical_verdict(_src, _p)
        _rows.append({"fixture": _name, "mechanical_ok": _v["mechanical_ok"],
                      "failed_stage": _v["failed_stage"],
                      "validate_errors": len(_p.get("validate_errors") or []),
                      "sweep_errors": len((_p.get("sweep") or {}).get("errors") or []),
                      "first_error": (_v["errors"] or [""])[0][:80]})
    print("harness self-test:",
          "PASSED" if all(r["mechanical_ok"] == _expected[r["fixture"]] for r in _rows)
          else "FAILED — do not trust the results below")
    display(pd.DataFrame(_rows))
else:
    print("reference notebook not found — self-test skipped")


## G. Source inspection, attempts, and the optional repair

`source_inspection` is a static `ast` read of the generated code: the numeric parameters the
model introduced (module-level constants and the numeric literals inside `transition_kernel`),
and the source of `is_terminal` — the actual terminal *logic*, not just the terminal states
that happened to be reached. **Nothing here judges or rewrites anything**; it makes invented
numbers visible in section J instead of leaving them buried in the code.

The optional repair round is **mechanical only**: it passes back the validator's own errors and
asks for the smallest correction. It never mentions stakeholders, uncertainty, observability or
horizon, so it cannot launder a semantic failure. First-pass and repaired outcomes are stored
and reported separately — the headline numbers in section I are first-pass.


In [ ]:
def source_inspection(code: str) -> dict:
    """Static read of the generated source. Records, never corrects."""
    info = {"module_constants": [], "kernel_numeric_literals": [], "is_terminal_source": None,
            "n_lines": len(code.splitlines()) if code else 0, "parse_error": None}
    if not code:
        return info
    try:
        tree = ast.parse(code)
    except SyntaxError as exc:
        info["parse_error"] = str(exc)
        return info

    for node in tree.body:                       # module-level parameters the model introduced
        if isinstance(node, ast.Assign):
            for target in node.targets:
                if isinstance(target, ast.Name):
                    info["module_constants"].append(f"{target.id} = {ast.unparse(node.value)}")

    for node in ast.walk(tree):
        if isinstance(node, ast.FunctionDef) and node.name == "transition_kernel":
            for sub in ast.walk(node):
                if isinstance(sub, ast.Constant) and isinstance(sub.value, (int, float)) \
                        and not isinstance(sub.value, bool) and 0 < sub.value < 1:
                    info["kernel_numeric_literals"].append(sub.value)
        if isinstance(node, ast.FunctionDef) and node.name == "is_terminal":
            info["is_terminal_source"] = ast.get_source_segment(code, node) or ast.unparse(node)
    return info


RECORDS_PATH = RUN_DIR / "records.jsonl"


def run_attempt(case: dict, variant: str, gen: int, repair: int,
                errors: str = "") -> dict:
    """One generation plus the full mechanical pipeline; appended to records.jsonl."""
    tag = f"{case['case_id'].replace('.', '')}_{variant}_g{gen}_r{repair}"
    system, user = build_prompt(case, variant)
    if repair:
        user += "\n\n" + REPAIR_INSTRUCTION.format(errors=errors[:3000])
    seed = SEED + 1000 * gen + repair

    try:
        raw, meta = GENERATOR.generate(system, user, seed)
    except Exception as exc:                                   # a backend failure is a result
        raw, meta = "", {"backend": BACKEND, "model_id": CONFIG["model_id"],
                         "input_tokens": None, "output_tokens": None, "seconds": None,
                         "seed": seed, "error": f"{type(exc).__name__}: {exc}"}
    code = extract_python(raw)

    (RUN_DIR / "prompts" / f"{tag}.md").write_text(f"# SYSTEM\n\n{system}\n\n# USER\n\n{user}\n")
    (RUN_DIR / "raw" / f"{tag}.txt").write_text(raw or "")

    module_path, probe = None, {"stage": "no_code", "ok": False, "error": {
        "type": "NoCode", "message": "no code in the response", "traceback": ""}}
    if code:
        module_path = write_module(code, RUN_DIR / "generated" / f"{tag}.py")
        probe = run_probe(module_path)
    (RUN_DIR / "probe" / f"{tag}.json").write_text(json.dumps(probe, indent=2, default=str))

    verdict = mechanical_verdict(code, probe)
    if not code and meta.get("error"):
        verdict["failed_stage"], verdict["errors"] = "backend_error", [meta["error"]]

    attempt = {"case_id": case["case_id"], "title": case["title"],
               "format_class": case["format_class"], "prompt_variant": variant,
               "generation": gen, "repair_round": repair, "stage": "repair" if repair else "first_pass",
               "tag": tag, "code_generated": bool(code), "code": code, "raw": raw,
               "meta": meta, "probe": probe, "verdict": verdict,
               "inspection": source_inspection(code),
               "module_path": str(module_path.relative_to(RUN_DIR)) if module_path else None}
    with RECORDS_PATH.open("a") as fh:            # stream, so a Colab disconnect loses nothing
        fh.write(json.dumps({k: v for k, v in attempt.items() if k != "raw"}, default=str) + "\n")
    return attempt


def run_cell(case: dict, variant: str, gen: int) -> list:
    """One (case, variant, generation): the first pass, then at most one repair."""
    attempts = [run_attempt(case, variant, gen, 0)]
    if (ENABLE_REPAIR and attempts[0]["code_generated"]
            and not attempts[0]["verdict"]["mechanical_ok"]):
        attempts.append(run_attempt(case, variant, gen, 1,
                                    errors="\n".join(attempts[0]["verdict"]["errors"])))
    return attempts


## H. Run

The grid is `case x prompt variant x generation`, first pass plus at most one repair. Each row
prints as it finishes and is appended to `records.jsonl`, so a Colab disconnect costs only the
generation in flight. Use `CASE_FILTER` / `VARIANT_FILTER` / `N_GENERATIONS` in section A to cut
the grid down while iterating.


In [ ]:
_grid = [(c, v, g) for c in CASES for v in PROMPT_VARIANTS for g in range(N_GENERATIONS)]
print(f"{len(CASES)} cases x {len(PROMPT_VARIANTS)} variants x {N_GENERATIONS} generations "
      f"= {len(_grid)} first-pass generations"
      + (" (+ repairs)" if ENABLE_REPAIR else "") + "\n")

ATTEMPTS, _t0 = [], time.time()
for _i, (_case, _variant, _gen) in enumerate(_grid, 1):
    for _a in run_cell(_case, _variant, _gen):
        ATTEMPTS.append(_a)
        _v, _m = _a["verdict"], _a["meta"]
        print(f"[{_i:>3}/{len(_grid)}] {_a['tag']:<28} "
              f"code={'y' if _a['code_generated'] else 'n'} "
              f"exec={'y' if _v['exec_ok'] else 'n'} "
              f"valid={'y' if _v['validate_ok'] else 'n'} "
              f"mech={'PASS' if _v['mechanical_ok'] else 'FAIL'} "
              f"{str(_m.get('seconds', '')):>6}s  {(_v['errors'] or [''])[0][:58]}")
print(f"\n{len(ATTEMPTS)} attempts in {(time.time() - _t0)/60:.1f} min")


## I. Results

Three tables, kept separate on purpose:

* **`first_pass_df`** — one row per `case x variant x generation`, before any repair. This is
  the table the prompt ablation is read from.
* **`variant_summary`** — the same rows aggregated per prompt variant: syntax, execution and
  validation rates, plus size and cost. Compare variants here, and remember that none of
  `P1`–`P4` is presumed to help.
* **`repair_df`** — only the cells that failed first pass, showing whether one mechanical
  repair fixed them and what it cost.

Structural columns (`n_players`, `n_state_vars`, `n_actions`, `max_depth`,
`n_probability_params`, …) are descriptive. They are *not* quality scores: a bigger model is
not a better model, and a model with more probability parameters may simply have invented more
numbers.


In [ ]:
def structure_signals(attempt: dict) -> dict:
    """Descriptive numbers about the generated model. Not scores."""
    st = (attempt["probe"] or {}).get("structure") or {}
    sw = (attempt["probe"] or {}).get("sweep") or {}
    insp = attempt["inspection"]
    if not st:
        return {k: None for k in ("n_players", "n_state_vars", "n_actions", "n_states_explored",
                                  "max_depth", "n_terminal_sampled", "n_probability_params",
                                  "n_kernel_literals", "distinct_probabilities", "caps_hit",
                                  "all_players_same_observations", "deterministic_only",
                                  "n_unused_actions")}
    obs = st.get("observations") or {}
    probs = sw.get("probabilities") or []
    return {
        "n_players": len(st.get("players") or {}),
        "n_state_vars": len(st.get("state_variables") or {}),
        "n_actions": len(st.get("action_catalogue") or {}),
        "n_states_explored": sw.get("n_states_explored"),
        "max_depth": sw.get("max_depth"),
        "n_terminal_sampled": sw.get("n_terminal_sampled"),
        "n_probability_params": len(insp["module_constants"]) + len(insp["kernel_numeric_literals"]),
        "n_kernel_literals": len(insp["kernel_numeric_literals"]),
        "distinct_probabilities": [p for p in probs if 0 < p < 1],
        "caps_hit": ",".join(sw.get("caps_hit") or []),
        "all_players_same_observations": (len(obs) > 1 and
                                          len({tuple(v) for v in obs.values()}) == 1),
        "deterministic_only": bool(probs) and not [p for p in probs if 0 < p < 1],
        "n_unused_actions": len(st.get("unused_catalogue_actions") or []),
    }


def attempt_row(a: dict) -> dict:
    v, m = a["verdict"], a["meta"]
    return {"case_id": a["case_id"], "format_class": a["format_class"],
            "prompt_variant": a["prompt_variant"], "generation": a["generation"],
            "stage": a["stage"], "repair_round": a["repair_round"],
            "model": m.get("model_id"), "seed": m.get("seed"),
            "code_generated": a["code_generated"], "syntax_ok": v["syntax_ok"],
            "exec_ok": v["exec_ok"], "validate_ok": v["validate_ok"], "sweep_ok": v["sweep_ok"],
            "mechanical_ok": v["mechanical_ok"], "failed_stage": v["failed_stage"],
            "validation_error": (v["errors"] or [""])[0][:300], "n_errors": len(v["errors"]),
            "input_tokens": m.get("input_tokens"), "output_tokens": m.get("output_tokens"),
            "truncated": m.get("truncated"), "seconds": m.get("seconds"),
            "class_name": (a["probe"] or {}).get("class_name"),
            "module_path": a["module_path"], "tag": a["tag"], **structure_signals(a)}


attempts_df = pd.DataFrame([attempt_row(a) for a in ATTEMPTS])
first_pass_df = attempts_df[attempts_df.repair_round == 0].reset_index(drop=True)

_rate = lambda s: round(100 * pd.to_numeric(s, errors="coerce").mean(), 1)   # NaN if unknown
variant_summary = (first_pass_df.groupby("prompt_variant").agg(
    n=("case_id", "size"),
    code_pct=("code_generated", _rate), syntax_pct=("syntax_ok", _rate),
    exec_pct=("exec_ok", _rate), validate_pct=("validate_ok", _rate),
    mechanical_pct=("mechanical_ok", _rate), truncated_pct=("truncated", _rate),
    med_players=("n_players", "median"), med_state_vars=("n_state_vars", "median"),
    med_actions=("n_actions", "median"), med_depth=("max_depth", "median"),
    med_prob_params=("n_probability_params", "median"),
    med_out_tokens=("output_tokens", "median"), med_seconds=("seconds", "median"),
).reset_index())

_repaired = attempts_df[attempts_df.repair_round > 0]
if len(_repaired):
    _keys = ["case_id", "prompt_variant", "generation"]
    repair_df = (first_pass_df[~first_pass_df.mechanical_ok][_keys + ["failed_stage",
                                                                     "validation_error"]]
                 .merge(_repaired[_keys + ["mechanical_ok", "failed_stage", "output_tokens"]],
                        on=_keys, how="left", suffixes=("_first", "_repaired")))
else:
    repair_df = pd.DataFrame()

attempts_df.to_csv(RUN_DIR / "attempts.csv", index=False)
first_pass_df.to_csv(RUN_DIR / "first_pass.csv", index=False)
variant_summary.to_csv(RUN_DIR / "variant_summary.csv", index=False)
if len(repair_df):
    repair_df.to_csv(RUN_DIR / "repair.csv", index=False)

print("FIRST PASS, by prompt variant (percentages of generations):")
display(variant_summary)


In [ ]:
print("first pass, per case x variant:")
display(first_pass_df[["case_id", "format_class", "prompt_variant", "generation",
                       "code_generated", "syntax_ok", "exec_ok", "validate_ok", "mechanical_ok",
                       "failed_stage", "n_players", "n_state_vars", "max_depth",
                       "n_probability_params"]])

print("\nwhere first-pass attempts failed:")
display(first_pass_df.groupby("failed_stage", dropna=False).size().rename("n").to_frame())

if len(repair_df):
    print("\nrepair outcomes (mechanical only, first pass kept separate):")
    display(repair_df)


In [ ]:
# the exact validator errors, uncut — these drive the next prompt iteration
for _r in first_pass_df[~first_pass_df.mechanical_ok].itertuples():
    _a = next(a for a in ATTEMPTS if a["tag"] == _r.tag)
    print(f"--- {_r.tag}  [{_r.failed_stage}]")
    for _e in _a["verdict"]["errors"][:4]:
        print("   ", _e[:200])


## J. Semantic inspection — human review

Levels 1 and 2 are decided by machine. Level 3 is not, and nothing here scores it. What this
section does is put everything a reviewer needs on one page per generated model:

* **players** and their descriptions — who did the model decide is affected?
* **state variables** — do they track what matters for those humans' future options?
* **actions per player** — what each player can actually do, as observed in the sweep;
* **observations per player** — who learns what, next to the state variables;
* **terminal logic** — the source of `is_terminal`, plus the terminal states reached;
* **introduced probability parameters** — every number the model put in, and where.

Below each model sits that case's own *should include* / *should NOT include* list from the
repository. The model never saw it.

The listed failure modes are visible here rather than fixed: a missing human shows up as an
absent player; an invented actor as a player the situation never implies; an arbitrary
probability as a literal in the kernel with nothing in the situation to support it; wrong
observability as an observation that reveals a latent variable, or as identical observations
for every player; an arbitrary horizon as terminal logic that counts steps.


In [ ]:
def review_sheet(case: dict, attempt: dict) -> str:
    st = (attempt["probe"] or {}).get("structure") or {}
    sw = (attempt["probe"] or {}).get("sweep") or {}
    v, insp, sig = attempt["verdict"], attempt["inspection"], structure_signals(attempt)
    L = [f"# {case['case_id']} — {case['title']}", "",
         f"*variant:* `{attempt['prompt_variant']}` · *generation:* {attempt['generation']} · "
         f"*stage:* {attempt['stage']} · *model:* `{attempt['meta'].get('model_id')}`",
         f"*mechanical:* **{'PASS' if v['mechanical_ok'] else 'FAIL'}**"
         + ("" if v["mechanical_ok"] else f" at `{v['failed_stage']}` — {(v['errors'] or [''])[0][:160]}"),
         f"*module:* `{attempt['module_path']}`  ·  *format class (data/README.md):* "
         f"{case['format_class']}", "",
         "## Situation (verbatim, as given to the model)", "", f"> {case['situation']}", ""]

    if not attempt["code_generated"]:
        L += ["## No model was produced", "", f"`{(v['errors'] or [''])[0]}`", ""]
    else:
        L += ["## Players", ""]
        L += [f"- `{p}` — {d}" for p, d in (st.get("players") or {}).items()] or ["- (none)"]
        L += ["", "## State variables", ""]
        for name, meta in (st.get("state_variables") or {}).items():
            desc = meta.get("description", "") if isinstance(meta, dict) else meta
            L.append(f"- `{name}` — {desc}")
        L += ["", f"Initial state: `{st.get('initial_state')}`", "", "## Actions", ""]
        for a, d in (st.get("action_catalogue") or {}).items():
            L.append(f"- `{a}` — {d}")
        L += ["", "Reachable per player (from the sweep):", ""]
        for p, acts in (st.get("actions_by_player") or {}).items():
            L.append(f"- `{p}`: {acts}")
        if st.get("unused_catalogue_actions"):
            L.append(f"- declared but never legal: `{st['unused_catalogue_actions']}`")
        L += ["", "## Observations, per player", ""]
        for p, obs in (st.get("observations") or {}).items():
            L.append(f"- `{p}`: {obs}")
        if sig["all_players_same_observations"]:
            L.append("- *(every player sees the same observations — check whether the situation "
                     "really is symmetric)*")
        L += ["", "## Terminal logic", "", "```python",
              (insp["is_terminal_source"] or "(not found)").strip(), "```", "",
              "States reached that were terminal:", ""]
        L += [f"- `{t}`" for t in (st.get("terminal_states") or [])] or \
             ["- none reached within the exploration caps"]
        L += ["", "## Introduced probability parameters", "",
              f"- module-level constants: `{insp['module_constants'] or 'none'}`",
              f"- numeric literals inside `transition_kernel`: "
              f"`{insp['kernel_numeric_literals'] or 'none'}`",
              f"- probability values observed in the sweep: `{sw.get('probabilities')}`",
              "", "*For each number: does the situation support it, or was it invented?*", "",
              "## Size", "",
              f"- {sig['n_players']} players · {sig['n_state_vars']} state variables · "
              f"{sig['n_actions']} actions",
              f"- {sig['n_states_explored']} states explored, depth {sig['max_depth']}, "
              f"{sw.get('n_transition_entries')} transition entries"
              + (f", caps hit: {sig['caps_hit']}" if sig["caps_hit"] else ""), ""]

    L += ["## Reviewer checklist — repository's *should include* (never shown to the model)", ""]
    L += [f"- [ ] {item}" for item in case["expect"]]
    L += ["", "## Reviewer checklist — repository's *should NOT include*", ""]
    L += [f"- [ ] {item}" for item in case["avoid"]]
    L += ["", "## Reviewer notes", "",
          "_(missing humans · invented actors or details · arbitrary probabilities · "
          "wrong observability · arbitrary horizon · missing actions or state variables)_", ""]
    return "\n".join(L)


for _a in ATTEMPTS:
    if _a["repair_round"] == 0:            # first-pass models are what gets reviewed
        _case = next(c for c in CASES if c["case_id"] == _a["case_id"])
        (RUN_DIR / "review" / f"{_a['tag']}.md").write_text(review_sheet(_case, _a))
print(f"wrote review sheets to {RUN_DIR / 'review'}")


In [ ]:
# one sheet inline; change the index to page through them
_first = [a for a in ATTEMPTS if a["repair_round"] == 0][0]
display(Markdown(review_sheet(next(c for c in CASES if c["case_id"] == _first["case_id"]),
                              _first)))


In [ ]:
# side-by-side across prompt variants for one case: what changed when the instruction changed?
_case_id = CASES[0]["case_id"]
_cmp = first_pass_df[first_pass_df.case_id == _case_id][
    ["prompt_variant", "generation", "mechanical_ok", "failed_stage", "n_players",
     "n_state_vars", "n_actions", "max_depth", "n_probability_params",
     "all_players_same_observations", "deterministic_only"]]
print(f"{_case_id} across prompt variants:")
display(_cmp)

for _t in first_pass_df[first_pass_df.case_id == _case_id].tag:
    _a = next(a for a in ATTEMPTS if a["tag"] == _t)
    _st = (_a["probe"] or {}).get("structure") or {}
    print(f"\n{_a['prompt_variant']}: players={list((_st.get('players') or {}))} "
          f"state_vars={list((_st.get('state_variables') or {}))}")


## K. What was produced, and what is still open

### Saved artefacts (everything under the run directory)

| Path | Contents |
|---|---|
| `config.json` | model, decoding settings, grid, caps for this run |
| `posg_abc.py` | the interface, as copied into this notebook |
| `probe.py` | the isolated validator/sweep |
| `prompts/*.md` | the exact system and user message of every call |
| `raw/*.txt` | raw model output, before code extraction |
| `generated/*.py` | executable modules (preamble + generated class) |
| `probe/*.json` | full probe output per attempt |
| `records.jsonl` | one line per attempt, written as it happens |
| `attempts.csv`, `first_pass.csv`, `variant_summary.csv`, `repair.csv` | the results tables |
| `review/*.md` | one review sheet per first-pass generation |

### What is measured, and what is not

Measured mechanically: code produced, syntax, execution (import + subclass + instantiate),
`validate()`, the joint-action sweep, exact errors, tokens, wall-clock, and descriptive size
numbers. Measured by a human, from section J: whether the world model is any good.

Not measured, deliberately: nothing scores semantic quality automatically, and no semantic
problem is corrected. The repair round only ever sees mechanical errors.

### Open questions this notebook exposes rather than settles

1. **Initial uncertainty.** The interface has a single `initial_state`, so an unresolved fact
   must be a state variable resolved in a transition. The JSON track allows a distribution over
   initial states. Whether the Python interface should gain an `initial_distribution` is a real
   design question — the review sheets show how models cope in the meantime.
2. **Passive stakeholders.** `validate()` requires every player to act in every non-terminal
   state, so an affected human who does nothing needs a no-op. Watch whether affected humans
   are modelled as players with real options or as no-op placeholders.
3. **Rewards.** The interface makes `rewards` abstract although this project derives value
   downstream. The prompt says the values are not evaluated; whether they should be in the
   interface at all is worth raising.
4. **Unsupported probabilities.** Every number a model introduces is recorded, never fixed. If
   a pattern emerges (a favourite 0.5, or arbitrary 0.3/0.7 splits), that is evidence for the
   next prompt iteration — and an argument about what the interface should require.
5. **Ablation power.** With `N_GENERATIONS = 1` the variant table is a smoke test, not a
   comparison: sampling noise at temperature 0.2 is comfortably larger than most differences
   between prompt variants. Raise `N_GENERATIONS` before drawing conclusions.


In [ ]:
print("run directory:", RUN_DIR)
for _p in sorted(RUN_DIR.rglob("*")):
    if _p.is_file():
        print(f"  {_p.relative_to(RUN_DIR)}  ({_p.stat().st_size} bytes)")
